In [ ]:
import pandas as pd 
from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.layers import Dense , Conv2D , MaxPooling2D , Flatten , Dropout , GlobalAveragePooling2D
from keras.models import Model
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.applications import MobileNetV2
import matplotlib.pyplot as plt
import json
from pathlib import Path

# Find the project root when the notebook is run from the project root or notebooks/.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "data" / "dogs.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root containing data/dogs.csv.")

DATA_DIR = PROJECT_ROOT / "data"
APP_DIR = PROJECT_ROOT / "app"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
%matplotlib widget

In [ ]:
df = pd.read_csv(DATA_DIR / "dogs.csv")

# Resolve CSV image paths against data/, regardless of the notebook's working directory.
def resolve_image_path(value):
    path = Path(str(value))
    if path.is_absolute() and path.is_file():
        return str(path.resolve())

    # Keep the dataset split directory when CSV entries contain older absolute paths.
    split_names = {"train", "test", "valid"}
    split_index = next((i for i, part in enumerate(path.parts) if part in split_names), None)
    relative_path = Path(*path.parts[split_index:]) if split_index is not None else path
    return str((DATA_DIR / relative_path).resolve())

df["filepaths"] = df["filepaths"].map(resolve_image_path)
missing_paths = [path for path in df["filepaths"] if not Path(path).is_file()]
if missing_paths:
    raise FileNotFoundError(
        f"{len(missing_paths)} dataset image(s) listed in data/dogs.csv were not found. "
        f"First missing path: {missing_paths[0]}"
    )

In [ ]:
train_df = df[df['data_label'] == 'train']

test_df = df[df['data_label'] == 'test']

valid_df = df[df['data_label']=='valid']

In [ ]:
print('Train :' ,len(train_df))
print('Test :' , len(test_df))
print('Valid :' , len(valid_df))

In [ ]:
train_generator = ImageDataGenerator(
    rescale = 1./255 , 
    rotation_range= 20 ,
    zoom_range = 0.2 ,
    horizontal_flip = True
)

In [ ]:
test_generator = ImageDataGenerator(
    rescale = 1./255
)

In [ ]:
img_size = 128
batchsize = 32

In [ ]:
train_data = train_generator.flow_from_dataframe(
    dataframe = train_df , 
    x_col = 'filepaths',
    y_col = 'labels' ,
    target_size = ( img_size , img_size),
    batch_size = batchsize,
    class_mode = 'categorical',
    shuffle=True
)

In [ ]:
valid_data = test_generator.flow_from_dataframe(
    dataframe = valid_df , 
    x_col = 'filepaths',
    y_col = 'labels' ,
    target_size = (img_size , img_size),
    batch_size = batchsize,
    class_mode = 'categorical'
)

In [ ]:
test_data = test_generator.flow_from_dataframe(
    dataframe = test_df , 
    x_col = 'filepaths',
    y_col = 'labels' ,
    target_size = (img_size , img_size),
    batch_size = batchsize,
    class_mode = 'categorical',
    shuffle = False
)

In [ ]:
num_classes = len(train_data.class_indices)

In [ ]:
# Save the class-to-index mapping alongside the application model.
class_indices_dict = train_data.class_indices
class_indices_path = APP_DIR / "class_indices.json"
with open(class_indices_path, "w", encoding="utf-8") as f:
    json.dump(class_indices_dict, f)
print(f"Class indices saved to {class_indices_path}")


## Transfer Learning

In [ ]:
# --- Initial configuration ---
IMG_SIZE = img_size  # Use the same image size as the data generators.
NUM_CLASSES = num_classes  # Number of classes inferred from the training data.
BATCH_SIZE = batchsize  # Use the same batch size as the data generators.

local_weights_path = WEIGHTS_DIR / "mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_128_no_top.h5"
if not local_weights_path.is_file():
    raise FileNotFoundError(f"Pretrained weights not found: {local_weights_path}")

# --- Build the model ---
# Load the MobileNetV2 base architecture without its original classification head.
base_model = MobileNetV2(
    weights=None,
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Load the locally stored pretrained weights.
base_model.load_weights(local_weights_path)

# Freeze the base model so only the new classification head is trained.
base_model.trainable = False

# Add the project-specific classification head.
x = base_model.output
x = GlobalAveragePooling2D()(x)  # Reduces overfitting compared with Flatten for this model.
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)  # Final class probabilities.

# Assemble the complete model.
model = Model(inputs=base_model.input, outputs=predictions)

# --- Compile the model ---
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model with MobileNetV2 created successfully!")
# model.summary()  # Uncomment to display the model details.


In [ ]:
# Save the best model checkpoint during training.
checkpoint = ModelCheckpoint(
    str(APP_DIR / "best_dog_breed_model.keras"),  # Save the model in app/.
    monitor='val_accuracy',       # Metric used to select the best checkpoint.
    save_best_only=True,          # Keep only the best checkpoint.
    mode='max',                   # Maximize validation accuracy.
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Train the model with early stopping and checkpointing.
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=50,  # EarlyStopping can stop training before this limit.
    callbacks=[early_stop, checkpoint],
    # batch_size=batchsize,  # Already configured in the image generators.
    # steps_per_epoch=...  # Set this only if a custom step count is needed.
)


In [ ]:
test_loss , test_acc = model.evaluate(test_data)
print(test_loss)
print(test_acc)